# Phase 3: Data Preparation

**CRISP-DM Phase Description:**  
This phase covers all activities to construct the final dataset from the initial raw data. Data preparation tasks are likely to be performed multiple times, and not in any prescribed order. This is typically the longest and most time-consuming phase of the CRISP-DM lifecycle.

---

In [40]:
# Standard library imports for this phase
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
%matplotlib inline

In [41]:
# Load the dataset from Phase 2 (update the path as needed)
DATA_PATH = 'path/to/your/data.csv'

TRAIN_PATH = '../data/raw/train.csv'
STORE_PATH = '../data/raw/store.csv'

df_train = pd.read_csv(TRAIN_PATH, low_memory=False)
df_store = pd.read_csv(STORE_PATH)

print(f"Train: {df_train.shape}")
print(f"Store: {df_store.shape}")

Train: (1017209, 9)
Store: (1115, 10)


---
### Task 1: Select Data

Decide on the data to be used for analysis. Consider which columns (features) and rows (records) to include or exclude based on:

- **Relevance:** Does this feature contribute to the data mining goal?
- **Data Quality:** Is the quality of this feature sufficient (e.g., too many missing values)?
- **Technical Constraints:** Are there limitations on data volume or specific feature types?

**Output:** A rationale for inclusion/exclusion of data, and the resulting subset.

**Instructions:** Select the columns and rows relevant to your analysis goal. Document your reasoning.

In [42]:
# TODO: Select the relevant columns and rows for your analysis.
# Document your rationale for including or excluding specific features.

# Rationale: We exclude days where the store was closed and the reason why is cause predicting 0 sales is trivial and that actually distorts the model's error metrics which is the RMSPE.
df_train = df_train[df_train['Open'] != 0]

# We drop the 'Open' afterward because it's now a constant which means that it's all 1
df_train = df_train.drop(['Open'], axis=1)

print(f"New Train Shape: {df_train.shape}")

New Train Shape: (844392, 8)


---
### Task 2: Clean Data

Raise data quality to the level required by the selected analysis techniques. Cleaning activities include:

- **Handle Missing Values:** Impute missing values (mean, median, mode, forward/backward fill) or remove rows/columns with excessive missing data.
- **Correct Errors:** Fix inaccurate or corrupted data entries.
- **Remove Duplicates:** Eliminate exact or near-duplicate records.
- **Handle Outliers:** Decide how to treat extreme values (keep, cap, transform, or remove).

**Instructions:** Apply appropriate cleaning techniques to address the data quality issues identified in Phase 2, Task 4.

In [43]:
# TODO: Handle missing values.
# Choose an appropriate strategy for each column.

# Strategy 1: Impute CompetitionDistance with the median
# Reason: The median is a solid measure that isn't pulled by outliers.
df_store['CompetitionDistance'] = df_store['CompetitionDistance'].fillna(df_store['CompetitionDistance'].median())

# Strategy 2: Impute CompetitionOpenSince columns with 0
# Reason: So if the data is missing, we can assume there is no known competition start date or that it is irrelevant.
df_store['CompetitionOpenSinceMonth'] = df_store['CompetitionOpenSinceMonth'].fillna(0)
df_store['CompetitionOpenSinceYear'] = df_store['CompetitionOpenSinceYear'].fillna(0)

# Strategy 3: Impute Promo2 variables with 0
# Reason: If not available then the store simply does not use 'Promo2' program.
df_store['Promo2SinceWeek'] = df_store['Promo2SinceWeek'].fillna(0)
df_store['Promo2SinceYear'] = df_store['Promo2SinceYear'].fillna(0)
df_store['PromoInterval'] = df_store['PromoInterval'].fillna(0)

print(f"Remaining Missing Values in Store Data: {df_store.isnull().sum().sum()}")

Remaining Missing Values in Store Data: 0


In [44]:
# TODO: Remove duplicate records.

before = len(df_train)
df_clean = df_train.drop_duplicates()
after = len(df_clean)

print(f"Removed {before - after} duplicate rows.")
print(f"Remaining: {after:,} rows.")

Removed 0 duplicate rows.
Remaining: 844,392 rows.


In [45]:
# TODO: Handle outliers.
# Choose a strategy: capping (winsorizing), removing, or transforming.

# We won't cap outliers since high-sales days are valid responses to Promotions (Q2). 
# Instead, we will actually use log transformation in Phase 4 to normalize the right-skewed distribution.

print("Strategy: Retaining all valid outliers to preserve Promotion impact patterns.")

highsales_days = len(df_clean[df_clean['Sales'] > 25000])
print(f"Number of high-sales days (>25k) preserved: {highsales_days}")

Strategy: Retaining all valid outliers to preserve Promotion impact patterns.
Number of high-sales days (>25k) preserved: 758


---
### Task 3: Construct Data (Feature Engineering)

This task involves creating new attributes (features) derived from existing ones that may be more useful for modelling. Common techniques include:

- **Derived Attributes:** Create new features from existing ones (e.g., extracting `year`, `month`, `day` from a datetime column; computing `total_spend = price * quantity`).
- **Binning / Discretisation:** Convert continuous variables into categorical bins (e.g., age groups).
- **Encoding Categorical Variables:** Convert categorical features into numerical representations (e.g., one-hot encoding, label encoding).
- **Scaling / Normalisation:** Scale numerical features to a common range (e.g., Min-Max scaling, Standardisation).

**Instructions:** Create new features or transform existing ones to improve model performance.

In [46]:
# TODO: Create derived attributes / new features.

# Extract components to capture seasonality patterns
df_clean['Date'] = pd.to_datetime(df_clean['Date'])
df_clean['Year'] = df_clean['Date'].dt.year
df_clean['Month'] = df_clean['Date'].dt.month
df_clean['Day'] = df_clean['Date'].dt.day
df_clean['WeekOfYear'] = df_clean['Date'].dt.isocalendar().week.astype(int)
df_clean['SalesPerCustomer'] = df_clean['Sales'] / df_clean['Customers']

# Quick note: Customers won't be available in the test set, but it's useful for the training insights.

print("Derived features were successfully created.")

Derived features were successfully created.


In [47]:
# TODO: Encode categorical variables.

# Example: One-hot encoding
# df_clean = pd.get_dummies(df_clean, columns=['categorical_col_1', 'categorical_col_2'], drop_first=True)

# Example: Label encoding for ordinal variables
# from sklearn.preprocessing import LabelEncoder
# le = LabelEncoder()
# df_clean['ordinal_col'] = le.fit_transform(df_clean['ordinal_col'])


from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

# We firstly handle StateHoliday to ensure it's all strings, then we encode and we treat '0' as a string so the encoder sees it as a category none.
df_clean['StateHoliday'] = df_clean['StateHoliday'].astype(str)
df_clean['StateHoliday'] = le.fit_transform(df_clean['StateHoliday'])

print("StateHoliday successfully encoded.")
print(f"Unique values are now: {df_clean['StateHoliday'].unique()}")

StateHoliday successfully encoded.
Unique values are now: [0 1 2 3]


In [48]:
# TODO: Scale / normalise numerical features if required.

print("Scaling skipped")
print("Reason: The tree-based models are sturdy to varying feature scales.")

Scaling skipped
Reason: The tree-based models are sturdy to varying feature scales.


---
### Task 4: Integrate Data

If your project uses multiple data sources, this task involves merging or combining them into a single, unified dataset. Activities include:

- **Merging Tables:** Join datasets on common keys (e.g., using `pd.merge()`).
- **Appending Records:** Concatenate datasets with the same structure (e.g., using `pd.concat()`).
- **Aggregation:** Summarise data at a different level of granularity.

**Instructions:** If using multiple data sources, merge or concatenate them below. If your project uses a single dataset, document that here and proceed to the next task.

In [49]:
# TODO: Integrate data from multiple sources (if applicable).

df_integrated = pd.merge(df_clean, df_store, on='Store', how='left')

print("INTEGRATION REPORT")
print(f"Final Row Count:    {df_integrated.shape[0]:,}")
print(f"Final Column Count: {df_integrated.shape[1]}")

missing_after_merge = df_integrated.isnull().sum().sum()
if missing_after_merge == 0:
    print("No missing values introduced by the merge.")
else:
    print(f"{missing_after_merge} missing values found post-merge.")

INTEGRATION REPORT
Final Row Count:    844,392
Final Column Count: 22
52 missing values found post-merge.


In [ ]:
# To fix the 52 missing values after the post-merge
df_integrated = df_integrated.ffill()
print(f"Final check for NAs: {df_integrated.isnull().sum().sum()}")

Final check for NAs: 0


In [53]:
# Optional: Verify the integrated data

cols_to_view = ['Date', 'Store', 'Sales', 'StoreType', 'Assortment', 'CompetitionDistance']
display(df_integrated[cols_to_view].head(10))

print("\nNew columns added from df_store:")
new_cols = [c for c in df_store.columns if c != 'Store']
print(new_cols)


,Date,Store,Sales,StoreType,Assortment,CompetitionDistance
0,2015-07-31,1,5263,c,a,1270.0
1,2015-07-31,2,6064,a,a,570.0
2,2015-07-31,3,8314,a,a,14130.0
3,2015-07-31,4,13995,c,c,620.0
4,2015-07-31,5,4822,a,a,29910.0
5,2015-07-31,6,5651,a,a,310.0
6,2015-07-31,7,15344,a,c,24000.0
7,2015-07-31,8,8492,a,a,7520.0
8,2015-07-31,9,8565,a,c,2030.0
9,2015-07-31,10,7185,a,a,3160.0



New columns added from df_store:
['StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval']


---
### Task 5: Format Data

This final preparation task ensures the data is in the correct format for the modelling tools. Activities include:

- **Data Type Conversions:** Ensure all columns have appropriate data types (e.g., numeric, datetime, categorical).
- **Column Reordering:** Arrange columns in a logical order (e.g., features first, target last).
- **Renaming:** Give columns clear, descriptive names.
- **Saving the Prepared Dataset:** Export the final, clean dataset for use in subsequent phases.

**Instructions:** Apply any final formatting changes and save the prepared dataset.

In [54]:
# TODO: Apply final formatting — data types, column order, renaming.

target_col = 'Sales'
feature_cols = [col for col in df_integrated.columns if col != target_col]
df_final = df_integrated[feature_cols + [target_col]]

df_final.columns = [col.lower() for col in df_final.columns] #renaming the columns to snake_case for PEP8 compliance

print("Columns renamed to lowercase and sales moved to the end.")

Columns renamed to lowercase and sales moved to the end.


In [55]:
# TODO: Verify the final prepared dataset.

print("=" * 60)
print("FINAL PREPARED DATASET SUMMARY")
print("=" * 60)
print(f"Shape: {df_final.shape[0]:,} rows x {df_final.shape[1]} columns")
print(f"Missing values: {df_final.isnull().sum().sum()}")
print(f"Duplicates: {df_final.duplicated().sum()}")
print(f"\nTarget Variable Range: {df_final['sales'].min()} to {df_final['sales'].max()}")
print(f"\nFinal Column List:")
print(df_final.columns.tolist())
display(df_final.head())

FINAL PREPARED DATASET SUMMARY
Shape: 844,392 rows x 22 columns
Missing values: 0
Duplicates: 0

Target Variable Range: 0 to 41551

Final Column List:
['store', 'dayofweek', 'date', 'customers', 'promo', 'stateholiday', 'schoolholiday', 'year', 'month', 'day', 'weekofyear', 'salespercustomer', 'storetype', 'assortment', 'competitiondistance', 'competitionopensincemonth', 'competitionopensinceyear', 'promo2', 'promo2sinceweek', 'promo2sinceyear', 'promointerval', 'sales']


,store,dayofweek,date,customers,promo,stateholiday,schoolholiday,year,month,day,weekofyear,salespercustomer,storetype,assortment,competitiondistance,competitionopensincemonth,competitionopensinceyear,promo2,promo2sinceweek,promo2sinceyear,promointerval,sales
0,1,5,2015-07-31,555,1,0,1,2015,7,31,31,9.482883,c,a,1270.0,9.0,2008.0,0,0.0,0.0,0,5263
1,2,5,2015-07-31,625,1,0,1,2015,7,31,31,9.702400,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct",6064
2,3,5,2015-07-31,821,1,0,1,2015,7,31,31,10.126675,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct",8314
3,4,5,2015-07-31,1498,1,0,1,2015,7,31,31,9.342457,c,c,620.0,9.0,2009.0,0,0.0,0.0,0,13995
4,5,5,2015-07-31,559,1,0,1,2015,7,31,31,8.626118,a,a,29910.0,4.0,2015.0,0,0.0,0.0,0,4822


In [56]:
# TODO: Save the prepared dataset for use in Phase 4 (Modelling).

# OUTPUT_PATH = 'path/to/your/prepared_data.csv'
# df_final.to_csv(OUTPUT_PATH, index=False)
# print(f"Prepared dataset saved to: {OUTPUT_PATH}")

import os
output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)

OUTPUT_PATH = os.path.join(output_dir, 'rossmann_prepared.csv')
df_final.to_csv(OUTPUT_PATH, index=False)

print(f"Prepared dataset successfully saved to: {OUTPUT_PATH}")

Prepared dataset successfully saved to: ../data/processed\rossmann_prepared.csv
